# NestedSimPy — Bank Renege

[NestedSimPy](https://nestedsimpy.github.io/) adapts SimPy's official [Bank Renege example](https://simpy.readthedocs.io/en/latest/examples/bank_renege.html) — a counter with reneging customers (a `Resource` plus condition events) — so the outer behavior is unchanged while inner simulations fork at each triggering event. This example uses `NestedResource`.

See the [example page](https://nestedsimpy.github.io/official-parity/bank-reneging.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile bank_reneging_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
Bank renege example (nested-sim adaptation).

Covers:

- Resources: Resource
- Condition events

Scenario:
  A counter with a random service time and customers who renege. Based on the
  program bank08.py from TheBank tutorial of SimPy 2. (KGM)
"""


RANDOM_SEED = 42
NEW_CUSTOMERS = 5  # Total number of customers
INTERVAL_CUSTOMERS = 10.0  # Generate new customers roughly every x seconds
MIN_PATIENCE = 1  # Min. customer patience
MAX_PATIENCE = 3  # Max. customer patience

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples", "bank_reneging")


def source(env, number, interval, counter):
    """Source generates customers randomly"""

    for i in range(number):
        c = customer(env, f"Customer{i:02d}", counter, time_in_bank=12.0)
        env.process(c)
        # Interarrival time: exponential with mean `interval`.
        yield env.nested_timeout(
            {"distribution": "exponential", "lambda": 1.0 / interval},
            label="interarrival",
        )


def customer(env, name, counter, time_in_bank):
    """Customer arrives, is served and leaves."""
    arrive = env.now
    print(f"{arrive:7.4f} {name}: Here I am")

    with counter.request() as req:
        patience = env.nested_timeout(
            {"distribution": "uniform", "low": MIN_PATIENCE, "high": MAX_PATIENCE},
            label="patience",
        )
        # Wait for the counter or abort at the end of our tether
        results = yield req | patience

        wait = env.now - arrive

        if req in results:
            # We got to the counter
            print(f"{env.now:7.4f} {name}: Waited {wait:6.3f}")
            yield env.nested_timeout(
                {"distribution": "exponential", "lambda": 1.0 / time_in_bank},
                label="service_time",
            )
            print(f"{env.now:7.4f} {name}: Finished")
        else:
            # We reneged
            print(f"{env.now:7.4f} {name}: RENEGED after {wait:6.3f}")


def main():
    """Run the bank reneging example in nested simulation and store the output files."""

    random.seed(RANDOM_SEED)
    env = NestedEnvironment()
    env._ns_print_branch_summary = False
    counter = NestedResource(env, capacity=1, nested_id="counter")

    env.process(source(env, NEW_CUSTOMERS, INTERVAL_CUSTOMERS, counter))

    # Output options first.
    env.set_output_options(out_dir=str(NESTED_OUTPUT_FOLDER), gzip_trace=False)
    env.set_post_processing_options(
        gzip_trace=False,
        package_latest=True,
        print_outputs=False,
        autoplot=DEFAULT_AUTOPLOT,
        autoplot_example="bank_reneging",
    )
    # Outer settings.
    env.set_rng("independent")
    env.set_outer_seed(RANDOM_SEED)
    env.set_triggering_objects(nested_id="counter")
    env.set_triggering_conditions({"on": "arrival", "frequency": 1})
    env.set_outer_stopping_condition(timeout=None, max_arrivals=None)
    # Inner settings.
    env.set_inner_repetitions(2)
    env.set_inner_stopping_condition(
        relative_time=None, triggering_customer_departs=True
    )

    env.nested_run()


if __name__ == "__main__":
    main()


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches fork).
!python bank_reneging_colab.py

## 3. Inspect the run

`OutputManager` reads the run folder and reports the triggering events, plots the outer trajectory, and exports the sample path.

In [ ]:
import glob, os
run = os.path.dirname(glob.glob("simpy_examples/bank_reneging/**/raw", recursive=True)[0])

from nestedsimpy import OutputManager
om = OutputManager(run)
print(f"{len(om.triggers())} triggering events; the outer path has {len(om.export_outer())} recorded events")

om.visualize_outer_static("outer.png")   # outer trajectory, saved as a static image
fig = om.visualize_outer_interactive()   # the same trajectory as an interactive plot
fig.show()

om.export_outer("outer.csv")      # the outer sample path as a CSV
print("wrote outer.csv")